In [1]:
import os
import numpy as np
import nibabel as nib
from neuromaps import datasets, resampling, stats, images, nulls

We define the paths to the aperiodic exponent data and the receptor map. Both variables are volumetric brain maps stored as nifti images. Additionally, we define a fake 'parcellation' to indicate the brain mask of the aperiodic exponent grid.

In [2]:
# Aperiodic exponent
project_root = r'C:\Users\Cristina\megtusalen'
apexp_file = os.path.join(project_root,'data','sources','apexp','FAM-001_rECpost_MEGMAG_apexp_clean.nii')

# Receptor map
dubois_file = datasets.fetch_annotation(source='dubois2015', return_single=True)


[References] Please cite the following papers if you are using this data:

  For {'source': 'dubois2015', 'desc': 'abp688', 'space': 'MNI152', 'res': '1mm'}:
  [primary]:
    Jonathan M DuBois, Olivier G Rousset, Jared Rowley, Manuel Porras-Betancourt, Andrew J Reader, Aurelie Labbe, Gassan Massarweh, Jean-Paul Soucy, Pedro Rosa-Neto, and Eliane Kobayashi. Characterization of age/sex and the regional distribution of mglur5 availability in the healthy human brain measured by high-resolution [11 c] abp688 pet. European journal of nuclear medicine and molecular imaging, 43(1):152–162, 2016.
    Justine Y Hansen, Golia Shafiei, Ross D Markello, Kelly Smart, Sylvia ML Cox, Martin Nørgaard, Vincent Beliveau, Yanjun Wu, Jean-Dominique Gallezot, Étienne Aumont, and others. Mapping neurotransmitter systems to the structural and functional organization of the human neocortex. Nature neuroscience, 25(11):1569–1581, 2022.
  [secondary]:
    


The receptor map has a much higher spatial resolution than the aperiodic exponent data. In this example the receptor map from Dubois2015 was defined in a 1mm resolution grid at the MNI coordinate system. Aperiodic exponents were derived from PSD fooof model fitting at each position of a 3d grid used in MEG source reconstruction. This results in a mismatch between resolutions, having the aperiodic exponent a much lower resolution than the receptor map.

In [4]:
dubois = images.load_nifti(dubois_file)
print("Receptor map shape:", dubois.shape)

apexp = images.load_data(apexp_file)
print("Aperiodic exponent shape:", apexp.shape)

Receptor map shape: (193, 229, 193)
Aperiodic exponent shape: (15, 19, 16)


In [5]:
# Resample dubois to the apexp grid (lower resolution)
dubois, apexp = resampling.resample_images(src=dubois_file, trg=apexp_file,
                                            src_space='MNI152',
                                            trg_space='MNI152',
                                            resampling='transform_to_trg')
# Now check the resolution of both maps
print("Resampled receptor map shape:", dubois.shape)
print("Aperiodic exponent shape:", apexp.shape)


C:\Users\Cristina\miniconda3\envs\pysurfer_env\lib\site-packages\neuromaps\transforms.py:254: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  out = nimage.resample_to_img(load_nifti(img), load_nifti(target),
C:\Users\Cristina\miniconda3\envs\pysurfer_env\lib\site-packages\nilearn\image\resampling.py:805: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  return resample_img(


Resampled receptor map shape: (15, 19, 16)
Aperiodic exponent shape: (15, 19, 16)


In [6]:
# Correlate dubois and apexp
corr = stats.compare_images(dubois, apexp)
print(f'Correlation: r = {corr:.04f}')

Correlation: r = 0.0299


To test whether this correlation is significant we will employ a spatial null model based on the variogram method. In order to speed up the process, we will use a brain mask to mask all the aperiodic exponents inside the brain. This mask is defined as a fake parcellation in a separate file. In this file, each position inside the brain contains a label from 1 to 2459.

In [8]:
 # Load brain mask ('parcellation')
parcellation_file = os.path.join(project_root,'data','sources','apexp','parcellation.nii')
parcellation_data = nib.load(parcellation_file)

# Get binary mask from all the values in parcellation_data that are non-zero
brain_mask = parcellation_data.get_fdata() != 0
brain_mask_flat = brain_mask.flatten()
print("Number of positions inside the brain: ", np.count_nonzero(brain_mask_flat))


Number of positions inside the brain:  2459


Now we extract receptor map values and aperiodic exponent values at the brain mask and we flatten them as vectors.

In [9]:
# Extract dubois data in the brain mask
dubois_data = images.load_data(dubois)
dubois_data_flat = dubois_data.flatten()
dubois_data_flat = dubois_data_flat[brain_mask_flat]

# Extract apexp data in the brain mask
apexp_data = images.load_data(apexp)
apexp_data_flat = apexp_data.flatten()
apexp_data_flat = apexp_data_flat[brain_mask_flat]

We compute rotated versions of the receptor map to create a null distribution of correlations in which spatial autocorrelation is preserved. We use the variogram methods. To make it more computationally efficient we only rotate and compute distances at the positions inside the brain. Is this correct?????

In [ ]:
# Compute the spatial nulls
n_perm = 1000
n_proc = 12
rotated = nulls.burt2020(dubois_data_flat, atlas = 'mni152', density = '2mm', parcellation=parcellation_data, n_perm=n_perm, n_proc=n_proc, seed=1234)

We have to embed the rotated versions of the receptor map in the original 3D grid to compute the neuromap stas.compare_images function.

In [ ]:
# Embed spatial nulls into the original 3d grid
nulls_4d = np.zeros(brain_mask.shape + (n_perm,), dtype=rotated.dtype)
nulls_4d.reshape(-1, n_perm)[brain_mask_flat, :] = rotated

Finally, we compute the spatial correlation and its p-value.

In [ ]:
# Compute correlation with pvalue
corr, pval = stats.compare_images(dubois, apexp, nulls=nulls_4d)
print(f'Correlation: r = {corr:.04f}, p = {pval:.04f}')